In [1]:
!git clone https://github.com/el-cer/SAVIA.git


Cloning into 'SAVIA'...
remote: Enumerating objects: 311, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 311 (delta 6), reused 11 (delta 4), pack-reused 290 (from 1)
Receiving objects: 100% (311/311), 5.08 MiB | 11.65 MiB/s, done.
Resolving deltas: 100% (126/126), done.


In [2]:
import google.colab
google.colab.drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls

drive  sample_data  SAVIA


In [4]:
!cd SAVIA/ && pip install -r requirements.txt --no-deps
!pip install numpy pandas requests pytest pytest-cov


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.6/250.6 kB 5.4 MB/s eta 0:00:00


In [5]:
!pip install pytest pytest-cov

In [6]:
!mkdir SAVIA/tests



In [7]:
import sys, os

repo_path = "/content/SAVIA"  # Remplace par le vrai chemin où se trouve "scripts"
sys.path.append(repo_path)
os.chdir(repo_path)

print("Chemin courant :", os.getcwd())
!ls scripts

Chemin courant : /content/SAVIA
call_api_classify.py  call_api.py  clean_classification_and_load.py  ETL.py


In [8]:
!touch scripts/__init__.py

In [9]:
%%writefile scripts/call_api.py

import requests
import json
import pandas as pd
import os

# === CONFIGURATION ===
API_URL = "https://saviapi.win/classify"
MODEL = "Mistral-7B-Instruct"

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
CSV_PATH = os.path.join(PROJECT_ROOT, "data", "silver", "tweets_cleaned_1.csv")


def call_api(text: str):
    """Appelle l’API de classification et renvoie la réponse JSON."""
    payload = {
        "model": MODEL,
        "input": text
    }
    try:
        response = requests.post(API_URL, json=payload)
        response.raise_for_status()
        return response.json(), response.status_code
    except requests.exceptions.RequestException as e:
        return {"error": str(e)}, 500


def main():
    """Lecture du CSV, classification d’un tweet, et affichage du résultat."""
    print(f"📂 Lecture du fichier : {CSV_PATH}")

    df = pd.read_csv(CSV_PATH)

    # 🔹 Exemple : prendre un tweet précis (à adapter)
    target_id = 1814368963826164172
    if target_id not in df["id"].values:
        print(f"⚠️ Aucun tweet trouvé avec l'id {target_id}")
        return

    tweet = df[df["id"] == target_id]["clean_text"].values[0]
    print(f"🗣️ Texte : {tweet}")

    result, status_code = call_api(tweet)

    print(f"✅ Statut HTTP : {status_code}")
    print(f"🧠 Résultat : {json.dumps(result, indent=2, ensure_ascii=False)}")


if __name__ == "__main__":
    main()


Overwriting scripts/call_api.py


In [10]:
%%writefile tests/test_call_api.py
import sys, os
import pytest
import pandas as pd
import requests
from unittest.mock import patch, MagicMock

#Ajout du chemin racine pour trouver 'scripts'
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

import scripts.call_api as call_api_module


@patch("requests.post")
def test_call_api_success(mock_post):
    """Test de la fonction call_api() avec réponse valide."""
    mock_response = MagicMock()
    mock_response.status_code = 200
    mock_response.json.return_value = {
        "label": "problème avéré",
        "domaine": "fixe",
        "sous_domaine": "réseau",
        "score": 0.95
    }
    mock_post.return_value = mock_response

    #Appel de la fonction directement
    result, status_code = call_api_module.call_api("Problème de connexion Internet à la maison")

    assert status_code == 200
    assert result["label"] == "problème avéré"
    assert result["domaine"] == "fixe"


Writing tests/test_call_api.py


In [11]:
!pytest tests/test_call_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/SAVIA
plugins: cov-7.0.0, langsmith-0.4.38, typeguard-4.4.4, anyio-4.11.0
collected 1 item                                                               

tests/test_call_api.py::test_call_api_success PASSED                     [100%]

============================== 1 passed in 0.80s ===============================


In [12]:
%%writefile scripts/call_api_classify.py
import requests
import os
import pandas as pd
from datetime import datetime
import time

API_URL = "https://saviapi.win/classify"
MODEL = "Mistral-7B-Instruct"

CLASSIFICATION_CONTEXT = (
    "Tu es un assistant expert du Service Après-Vente (SAV) de l’opérateur Free. "
    "Ton objectif est d'analyser un tweet client et de déterminer s’il décrit un problème technique réel ou non, "
    "puis de classer ce problème par domaine et sous-domaine.\n\n"
    "⚠️ Ne considère pas comme 'problème avéré' les tweets humoristiques, ironiques, publicitaires, "
    "ou ceux qui ne contiennent aucun signe de plainte réelle ou technique.\n\n"
    "🔸 Le résultat doit toujours être un JSON unique au format :\n"
    "{\"label\": label, \"domaine\": domaine, \"sous_domaine\": sous_domaine, \"score\": score}\n\n"
    "🔹 'score' = niveau de confiance entre 0.0 et 1.0.\n\n"
    "🔹 Choix possibles :\n"
    "- label : ['problème avéré', 'problème non avéré']\n"
    "- domaine : ['mobile', 'fixe', 'facture', 'contact']\n"
    "- sous_domaine : ['réseau', 'wifi', 'box', 'appel voix', 'sécurité', 'autres']\n\n"
    "🧭 Guide de décision :\n"
    "- Si le tweet demande un **conseiller, une aide, ou mentionne une panne/dysfonctionnement**, choisis 'problème avéré'.\n"
    "- Si le message est **vague, court, ironique ou sans signe de problème**, choisis 'problème non avéré'.\n"
    "- Si le texte parle de **connexion Internet, lenteur, perte de signal**, choisis domaine='fixe', sous_domaine='réseau' ou 'wifi'.\n"
    "- Si le texte parle de **carte SIM, 4G, 5G, appels, SMS**, choisis domaine='mobile', sous_domaine='appel voix' ou 'réseau'.\n"
    "- Si le texte évoque **facture, prélèvement, paiement, compte client**, choisis domaine='facture'.\n"
    "- Si le texte mentionne **mot de passe, piratage, sécurité**, choisis sous_domaine='sécurité'.\n"
    "- Si le tweet est un **mème, une blague, ou hors sujet technique**, choisis toujours 'problème non avéré'.\n\n"
    "🧠 Analyse le texte avec bon sens. Ne crée jamais de nouvelles catégories. "
    "Sois sobre, rigoureux et évite toute surclassification."
)

def classify_text(row, retries=3):
    """Classe un tweet donné avec gestion des erreurs et du temps."""
    tweet = str(row["clean_text"])
    payload = {"prompt": tweet, "model": MODEL, "context": CLASSIFICATION_CONTEXT}
    start = time.perf_counter()

    for attempt in range(retries):
        try:
            response = requests.post(API_URL, json=payload, timeout=60)
            if response.status_code == 200:
                data = response.json()
                return {
                    "id": row["id"],
                    "label": data.get("label", "Unknown"),
                    "domaine": data.get("domaine", "Unknown"),
                    "sous_domaine": data.get("sous_domaine", "Unknown"),
                    "score": data.get("score", None),
                    "status_code": response.status_code,
                    "duration_seconds": round(time.perf_counter() - start, 3)
                }
            else:
                print(f"⚠️ Erreur API : {response.status_code}, tentative {attempt+1}/3")
        except requests.exceptions.RequestException as e:
            print(f"⚠️ {e} (tentative {attempt+1}/3)")
            time.sleep(2)

    return {"id": row["id"], "label": "Error", "status_code": "ERROR"}

if __name__ == "__main__":
    INPUT_CSV = "../data/silver/tweets_cleaned_1.csv"
    OUTPUT_CSV = "../data/gold/tweets_classified_2.csv"

    print(f"{datetime.now():%Y-%m-%d %H:%M:%S} - Début de la classification")
    df = pd.read_csv(INPUT_CSV)

    results = [classify_text(row) for _, row in df.iterrows()]
    df_out = pd.merge(df, pd.DataFrame(results), on="id", how="left")

    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    print(f"✅ Export terminé vers : {OUTPUT_CSV}")


Overwriting scripts/call_api_classify.py


In [13]:
%%writefile tests/test_call_api_classify.py
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

import pytest
from unittest.mock import patch, MagicMock
import pandas as pd
import requests
import scripts.call_api_classify as classify_module

@patch("requests.post")
def test_classify_text_success(mock_post):
    """Test normal : l'API renvoie un JSON valide et un code 200."""
    mock_response = MagicMock()
    mock_response.status_code = 200
    mock_response.json.return_value = {
        "label": "problème avéré",
        "domaine": "fixe",
        "sous_domaine": "wifi",
        "score": 0.93
    }
    mock_post.return_value = mock_response

    row = pd.Series({"id": 1, "clean_text": "Ma box ne fonctionne plus"})
    result = classify_module.classify_text(row)

    assert result["label"] == "problème avéré"
    assert result["domaine"] == "fixe"
    assert result["status_code"] == 200
    assert "duration_seconds" in result

@patch("requests.post", side_effect=requests.exceptions.RequestException("Timeout"))
def test_classify_text_api_error(mock_post):
    """Test erreur de connexion : la fonction doit renvoyer label=Error."""
    row = pd.Series({"id": 2, "clean_text": "Impossible d'appeler"})
    result = classify_module.classify_text(row, retries=1)
    assert result["label"] == "Error"
    assert result["status_code"] == "ERROR"

Writing tests/test_call_api_classify.py


In [14]:
!pytest -v tests/test_call_api_classify.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/SAVIA
plugins: cov-7.0.0, langsmith-0.4.38, typeguard-4.4.4, anyio-4.11.0
collected 2 items                                                              

tests/test_call_api_classify.py::test_classify_text_success PASSED       [ 50%]
tests/test_call_api_classify.py::test_classify_text_api_error PASSED     [100%]

============================== 2 passed in 2.55s ===============================


In [15]:
# tests/test_clean_classification_and_load.py
%%writefile tests/test_clean_classification_and_load.py
import pytest
import pandas as pd
from scripts import clean_classification_and_load as cleaner

# --- TEST strip_accents ---
def test_strip_accents_removes_accents_and_lowercases():
    assert cleaner.strip_accents("Éléphant") == "elephant"
    assert cleaner.strip_accents("RÉSEAU") == "reseau"
    assert cleaner.strip_accents("  Sécurité  ") == "securite"

# --- TEST pick_first_token ---
def test_pick_first_token_extracts_correct_value():
    assert cleaner.pick_first_token("wifi/internet") == "wifi"
    assert cleaner.pick_first_token("['réseau']") == "réseau"
    assert cleaner.pick_first_token("box, wifi") == "box"
    assert cleaner.pick_first_token("") == ""

# --- TEST normalize_label ---
def test_normalize_label_standardizes_labels():
    assert cleaner.normalize_label("probleme avere") == "problème avéré"
    assert cleaner.normalize_label("aucun") == "problème non avéré"
    assert cleaner.normalize_label("randomtext") == "inconnu"

# --- TEST normalize_domaine ---
def test_normalize_domaine_maps_synonyms():
    assert cleaner.normalize_domaine("internet") == "fixe"
    assert cleaner.normalize_domaine("factures") == "facture"
    assert cleaner.normalize_domaine("autre") == "inconnu"

# --- TEST normalize_sousdom ---
def test_normalize_sousdom_maps_correctly():
    assert cleaner.normalize_sousdom("wifi/internet") == "wifi"
    assert cleaner.normalize_sousdom("appel") == "appel voix"
    assert cleaner.normalize_sousdom("random") == "inconnu"

# --- TEST enforce_rules ---
def test_enforce_rules_sets_aucun_for_non_avere():
    row = {"label": "problème non avéré", "domaine": "fixe", "sous_domaine": "wifi"}
    new_row = cleaner.enforce_rules(row.copy())
    assert new_row["domaine"] == "aucun"
    assert new_row["sous_domaine"] == "aucun"

def test_enforce_rules_keeps_values_for_avere():
    row = {"label": "problème avéré", "domaine": "fixe", "sous_domaine": "wifi"}
    new_row = cleaner.enforce_rules(row.copy())
    assert new_row["domaine"] == "fixe"

# --- TEST quality_flags ---
def test_quality_flags_identifies_out_of_vocab_values():
    row = {"label": "badlabel", "domaine": "mobile", "sous_domaine": "wifi"}
    assert "label_oov" in cleaner.quality_flags(row)

    row = {"label": "problème avéré", "domaine": "bad", "sous_domaine": "wifi"}
    assert "domaine_oov" in cleaner.quality_flags(row)

    row = {"label": "problème avéré", "domaine": "mobile", "sous_domaine": "bad"}
    assert "sous_domaine_oov" in cleaner.quality_flags(row)

    row = {"label": "problème avéré", "domaine": "mobile", "sous_domaine": "wifi"}
    assert cleaner.quality_flags(row) == ""

# --- TEST main() (mocked IO) ---
def test_main_creates_output_file(tmp_path, monkeypatch):
    # Créer un mini CSV d’entrée simulé
    input_csv = tmp_path / "tweets_classified_2.csv"
    output_csv = tmp_path / "tweets_classified_clean_1.csv"

    df = pd.DataFrame([
        {"id": 1, "label": "probleme avere", "domaine": "internet", "sous_domaine": "wifi/internet"}
    ])
    df.to_csv(input_csv, index=False)

    monkeypatch.setattr(cleaner, "INPUT_CSV", str(input_csv))
    monkeypatch.setattr(cleaner, "OUTPUT_CSV", str(output_csv))

    cleaner.main()

    assert output_csv.exists(), "Le fichier nettoyé n'a pas été créé."

    df_out = pd.read_csv(output_csv)
    assert df_out.loc[0, "label"] == "problème avéré"
    assert df_out.loc[0, "domaine"] == "fixe"
    assert df_out.loc[0, "sous_domaine"] == "wifi"


Writing tests/test_clean_classification_and_load.py


In [16]:
!pytest tests/test_call_api_classify.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/SAVIA
plugins: cov-7.0.0, langsmith-0.4.38, typeguard-4.4.4, anyio-4.11.0
collected 2 items                                                              

tests/test_call_api_classify.py::test_classify_text_success PASSED       [ 50%]
tests/test_call_api_classify.py::test_classify_text_api_error PASSED     [100%]

============================== 2 passed in 2.55s ===============================


In [17]:
%%writefile tests/test_etl_clean_tweets.py
import sys, os
import pytest
import pandas as pd
import numpy as np
from unittest.mock import patch, MagicMock
from io import StringIO

# Ajout du chemin racine pour trouver le module ETL
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from scripts import ETL as etl


# --- UNIT TESTS ---

def test_clean_text_basic_cases():
    """Le texte doit être nettoyé de mentions, hashtags, URLs et emojis."""
    assert etl.clean_text("@user Salut #test http://site.com 😎!") == "salut"
    assert etl.clean_text("Texte normal") == "texte normal"
    assert etl.clean_text(None) == ""
    assert etl.clean_text("Ceci! contient? des; symboles...") == "ceci contient des symboles"


def test_normalize_text_removes_accents_and_lowercases():
    assert etl.normalize_text("Éléphant") == "elephant"
    assert etl.normalize_text("FREE_OFFICIEL") == "free_officiel"
    assert etl.normalize_text(None) == ""


def test_text_length_counts_correctly():
    assert etl.text_length("hello") == 5
    assert etl.text_length("") == 0
    assert etl.text_length(None) == 0


# --- INTEGRATION TEST ---

def test_pipeline_excludes_official_accounts(tmp_path, monkeypatch):
    """Vérifie qu'on exclut bien les comptes officiels et qu'on nettoie correctement."""

    # Création d'un mini DataFrame simulant le CSV brut
    df_raw = pd.DataFrame([
        {"id": 1, "created_at": "2024-01-01", "screen_name": "Free", "name": "Free", "full_text": "@user Bonjour", "in_reply_to": np.nan},
        {"id": 2, "created_at": "2024-01-02", "screen_name": "client", "name": "Jean", "full_text": "Problème de wifi 😡", "in_reply_to": np.nan},
        {"id": 3, "created_at": "2024-01-03", "screen_name": "client2", "name": "Paul", "full_text": "", "in_reply_to": None}
    ])

    # Remplace la lecture CSV et les chemins par des versions temporaires
    input_csv = tmp_path / "raw.csv"
    output_csv = tmp_path / "silver.csv"
    df_raw.to_csv(input_csv, index=False)

    monkeypatch.setattr(etl, "RAW_PATH", str(input_csv))
    monkeypatch.setattr(etl, "SILVER_PATH", str(output_csv))

    # Simulation d’exécution du pipeline (jusqu’à l’export)
    etl.df = pd.read_csv(etl.RAW_PATH)
    etl.df["clean_text"] = etl.df["full_text"].apply(etl.clean_text)
    etl.df["screen_name_clean"] = etl.df["screen_name"].apply(etl.normalize_text)
    etl.df["name_clean"] = etl.df["name"].apply(etl.normalize_text)
    etl.df["text_length"] = etl.df["clean_text"].apply(etl.text_length)

    df_clean = etl.df[
        ~etl.df["screen_name_clean"].isin(etl.EXCLUDED_ACCOUNTS)
        & ~etl.df["name_clean"].isin(etl.EXCLUDED_ACCOUNTS)
        & etl.df["in_reply_to"].isna()
    ].copy()

    df_clean = df_clean[df_clean["text_length"] > 0]
    df_clean.to_csv(output_csv, index=False)

    # Vérification : 1 ligne exclue (le compte "Free")
    df_out = pd.read_csv(output_csv)
    assert len(df_out) == 1
    assert "wifi" in df_out.iloc[0]["clean_text"]
    assert "😡" not in df_out.iloc[0]["clean_text"]


Writing tests/test_etl_clean_tweets.py


In [18]:
!ls tests

__pycache__		   test_clean_classification_and_load.py
test_call_api_classify.py  test_etl_clean_tweets.py
test_call_api.py


In [19]:
!touch scripts/__init__.py

In [20]:
!pytest tests/test_etl_clean_tweets.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/SAVIA
plugins: cov-7.0.0, langsmith-0.4.38, typeguard-4.4.4, anyio-4.11.0
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
_______________ ERROR collecting tests/test_etl_clean_tweets.py ________________
tests/test_etl_clean_tweets.py:11: in <module>
    from scripts import ETL as etl
scripts/ETL.py:31: in <module>
    df = pd.read_csv(RAW_PATH)
         ^^^^^^^^^^^^^^^^^^^^^
/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py:1026: in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py:620: in _read
    parser = TextFileReader(filepath_or_buffer, 

In [21]:
!pwd
!ls

/content/SAVIA
dashboard  frontend  notebooks	README.md	  scripts
data	   images    quality	requirements.txt  tests
